# 🚗 Initial Phase Number Plate Detection + Auto-Annotation

In [2]:
# Cell 1 - Install
!pip install ultralytics datasets -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 28.4 MB/s eta 0:00:0000:01
GPU: Tesla P100-PCIE-16GB


In [4]:
# Cell 2 - Fixed paths (no unzip needed)
import shutil
from pathlib import Path

SRC1 = Path("/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data")
SRC2 = Path("/kaggle/input/datasets/abdullahaamir13/plates-model3/batch02-model3/obj_train_data")

print("Batch01:", len(list(SRC1.glob("*.jpg"))), "images")
print("Batch02:", len(list(SRC2.glob("*.jpg"))), "images")

Batch01: 50 images
Batch02: 50 images


In [7]:
# Cell 3 - Direct Parquet download
import pandas as pd
import requests
from pathlib import Path
from PIL import Image
import io

# Download parquet files directly
train_url = "https://huggingface.co/datasets/keremberke/license-plate-object-detection/resolve/refs%2Fconvert%2Fparquet/full/train/0000.parquet"
val_url   = "https://huggingface.co/datasets/keremberke/license-plate-object-detection/resolve/refs%2Fconvert%2Fparquet/full/validation/0000.parquet"

!wget -q "{train_url}" -O /kaggle/working/train.parquet
!wget -q "{val_url}"   -O /kaggle/working/val.parquet

train_df = pd.read_parquet("/kaggle/working/train.parquet")
val_df   = pd.read_parquet("/kaggle/working/val.parquet")

print(f"Train rows: {len(train_df)}")
print(f"Val rows:   {len(val_df)}")
print(f"Columns: {train_df.columns.tolist()}")

Train rows: 6176
Val rows:   1765
Columns: ['image_id', 'image', 'width', 'height', 'objects']


In [8]:
# Cell 4 - Convert Parquet to YOLO format
import pandas as pd
from pathlib import Path
from PIL import Image
import io

def convert_parquet_to_yolo(df, split_name, out_dir):
    img_dir = Path(out_dir) / "images" / split_name
    lbl_dir = Path(out_dir) / "labels" / split_name
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)

    for i, row in df.iterrows():
        # Save image
        img_bytes = row['image']['bytes']
        img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
        W, H = img.size
        img.save(img_dir / f"plate_{i:05d}.jpg")

        # Save label
        with open(lbl_dir / f"plate_{i:05d}.txt", "w") as f:
            for bbox in row['objects']['bbox']:
                x, y, w, h = bbox
                cx = min(max((x + w/2) / W, 0), 1)
                cy = min(max((y + h/2) / H, 0), 1)
                bw = min(max(w / W, 0), 1)
                bh = min(max(h / H, 0), 1)
                f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")

    print(f"{split_name}: {len(df)} images saved")

convert_parquet_to_yolo(train_df, "train",      "/kaggle/working/plates_final")
convert_parquet_to_yolo(val_df,   "validation", "/kaggle/working/plates_final")

train: 6176 images saved
validation: 1765 images saved


In [9]:
# Cell 5 - Add your 100 Pakistani plates
import shutil
from pathlib import Path

SRC1      = Path("/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data")
SRC2      = Path("/kaggle/input/datasets/abdullahaamir13/plates-model3/batch02-model3/obj_train_data")
TRAIN_IMG = Path("/kaggle/working/plates_final/images/train")
TRAIN_LBL = Path("/kaggle/working/plates_final/labels/train")

added = 0
for src in [SRC1, SRC2]:
    for img in src.glob("*.jpg"):
        shutil.copy(img, TRAIN_IMG / f"pak_{img.name}")
        lbl = img.with_suffix(".txt")
        if lbl.exists():
            shutil.copy(lbl, TRAIN_LBL / f"pak_{lbl.name}")
            added += 1

print(f"Pakistani plates added: {added}")
print(f"Total train: {len(list(TRAIN_IMG.glob('*.jpg')))}")
print(f"Total val:   {len(list(Path('/kaggle/working/plates_final/images/validation').glob('*.jpg')))}")

Pakistani plates added: 100
Total train: 6276
Total val:   1765


In [10]:
# Cell 6 - dataset.yaml
import yaml

config = {
    "path": "/kaggle/working/plates_final",
    "train": "images/train",
    "val":   "images/validation",
    "nc": 1,
    "names": {0: "number_plate"}
}

with open("/kaggle/working/plates_final/dataset.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

!cat /kaggle/working/plates_final/dataset.yaml

names:
  0: number_plate
nc: 1
path: /kaggle/working/plates_final
train: images/train
val: images/validation


In [11]:
# Cell 7 - Train
!yolo task=detect \
      mode=train \
      model=yolov8s.pt \
      data=/kaggle/working/plates_final/dataset.yaml \
      epochs=50 \
      imgsz=640 \
      batch=16 \
      patience=15 \
      name=plates_model3 \
      project=/kaggle/working/runs \
      device=0 \
      exist_ok=True \
      cos_lr=True \
      mosaic=1.0

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/plates_final/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=

In [12]:
# Final cell - zip all outputs
import shutil

shutil.make_archive(
    "/kaggle/working/plates_model3_results",
    "zip",
    "/kaggle/working/runs/plates_model3"
)
print("Done. Download plates_model3_results.zip from Output tab.")

Done. Download plates_model3_results.zip from Output tab.


In [21]:
# Cell 8 - Find and load best.pt from output dataset
import os, zipfile
from pathlib import Path

# Check what's available
for root, dirs, files in os.walk("/kaggle/input"):
    for f in files:
        print(os.path.join(root, f))

/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj.names
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj.data
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/train.txt
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/03210000104.jpg
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/000000017.jpg
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/03210000109.txt
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/03210000058.txt
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/000000010.txt
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/03210000037.txt
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_data/03210000006.jpg
/kaggle/input/datasets/abdullahaamir13/plates-model3/batch01-model3/obj_train_

In [3]:
# Cell 8 - Load model (no extraction needed!)
from ultralytics import YOLO
from pathlib import Path

model = YOLO("/kaggle/input/datasets/abdullahaamir13/plates-model03-results/weights/best.pt")

BUS_IMAGES = Path("/kaggle/input/datasets/abdullahaamir13/busesnpmodel03/NOT")
print(f"Model loaded!!!")
print(f"Bus images found: {len(list(BUS_IMAGES.glob('*.jpg')))}")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Model loaded!!!
Bus images found: 1544


In [4]:
# Cell 9 - Auto-annotate
import shutil
from pathlib import Path

OUTPUT_DIR = Path("/kaggle/working/plate_auto_annotations")
IMG_OUT    = OUTPUT_DIR / "obj_train_data"
IMG_OUT.mkdir(parents=True, exist_ok=True)

results = model.predict(
    source=str(BUS_IMAGES),
    conf=0.30,
    iou=0.45,
    imgsz=640,
    save=False,
    verbose=False
)

detected = no_plate = 0

for r in results:
    img_path = Path(r.path)
    lbl_path = IMG_OUT / (img_path.stem + ".txt")
    shutil.copy(img_path, IMG_OUT / img_path.name)

    if r.boxes is None or len(r.boxes) == 0:
        lbl_path.touch()
        no_plate += 1
        continue

    with open(lbl_path, "w") as f:
        for box in r.boxes:
            cx, cy, bw, bh = box.xywhn[0].tolist()
            f.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
    detected += 1

print(f"Plates detected:  {detected}")
print(f"No plate found:   {no_plate}")
print(f"Detection rate:   {detected/(detected+no_plate)*100:.1f}%")

WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs



Corrupt JPEG data: 1 extraneous bytes before marker 0xd9


Plates detected:  1011
No plate found:   533
Detection rate:   65.5%


In [5]:
# Cell 10 - Package for CVAT
with open(OUTPUT_DIR / "obj.names", "w") as f:
    f.write("number_plate\n")

with open(OUTPUT_DIR / "obj.data", "w") as f:
    f.write("classes = 1\ntrain = train.txt\nnames = obj.names\n")

images = sorted(IMG_OUT.glob("*.jpg"))
with open(OUTPUT_DIR / "train.txt", "w") as f:
    for img in images:
        f.write(f"obj_train_data/{img.name}\n")

shutil.make_archive("/kaggle/working/plates_bus_auto_cvat", "zip", OUTPUT_DIR)
print(f"Total images: {len(images)}")
print("Download plates_bus_auto_cvat.zip from Output tab!!!")

Total images: 1544
Download plates_bus_auto_cvat.zip from Output tab!!!


# 🚗 Number Plate Detection + OCR Model04

**Only 2 datasets needed:**
| Dataset | Role |
|---|---|
| `abdullahaamir13/numberplatesmodel03` | All training data (batch03 annotated + model3 train/val) |
| `abdullahaamir13/plates-model03-results` | Model03 best.pt — fine-tune starting point |

**Pipeline:** Merge data → validate → fine-tune YOLOv8s → OCR with EasyOCR → batch inference → export

### Retraining

In [1]:
# Cell 1 — Install
!pip install ultralytics easyocr -q
import torch
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"CUDA: {torch.version.cuda}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 31.9 MB/s eta 0:00:0000:01
GPU : Tesla P100-PCIE-16GB
CUDA: 12.6


## Step 1 Verify Inputs

In [14]:
# Cell 2 — Check all paths before doing anything
import os
from pathlib import Path

# ── Dataset 1: numberplatesmodel03 ───────────────────────────────────────────
BASE       = Path("/kaggle/input/datasets/abdullahaamir13/numberplatesmodel03")

BATCH03    = BASE / "batch03-model3/obj_train_data"   # corrected annotations
M3_TRAIN   = BASE / "model3/model3/train"             # YOLO train split
M3_VAL     = BASE / "model3/model3/val"               # YOLO val split

# ── Dataset 2: plates-model03-results ────────────────────────────────────────
MODEL03_PT = Path("/kaggle/input/datasets/abdullahaamir13/plates-model03-results/weights/best.pt")

checks = {
    "batch03 corrected annotations": BATCH03,
    "model3 YOLO train split      ": M3_TRAIN,
    "model3 YOLO val split        ": M3_VAL,
    "Model03 best.pt (weights)    ": MODEL03_PT,
}

all_ok = True
for name, p in checks.items():
    if p.is_file():
        print(f"  ✅  {name}  (file)")
    elif p.is_dir():
        n_jpg = len(list(p.glob("*.jpg")))
        n_txt = len(list(p.glob("*.txt")))
        print(f"  ✅  {name}  ({n_jpg} images, {n_txt} labels)")
    else:
        print(f"  ❌  {name}  MISSING → {p}")
        all_ok = False

print()
if all_ok:
    print("✅ All inputs found — ready to proceed!")
else:
    print("❌ Fix missing paths above. Check your Kaggle input panel.")

  ✅  batch03 corrected annotations  (1119 images, 1119 labels)
  ✅  model3 YOLO train split        (0 images, 0 labels)
  ✅  model3 YOLO val split          (0 images, 0 labels)
  ✅  Model03 best.pt (weights)      (file)

✅ All inputs found — ready to proceed!


In [15]:
# Cell 3 — Explore exact folder structure (run once to confirm paths)
import os
from pathlib import Path

BASE = Path("/kaggle/input/datasets/abdullahaamir13/numberplatesmodel03")
for root, dirs, files in os.walk(BASE):
    depth = root.replace(str(BASE), "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}{Path(root).name}/")
    if depth < 3:
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... ({len(files)} files total)")

numberplatesmodel03/
  batch03-model3/
    obj.names
    obj.data
    train.txt
    obj_train_data/
      0904_bus_0.txt
      01398_bus_0.txt
      0976_bus_0.jpg
      01134_bus_0.jpg
      01230_bus_0.txt
      ... (2238 files total)
  model3/
    model3/
      val/
        labels/
        images/
      train/
        labels/
        images/


## Step 2 Build YOLO Dataset

In [1]:
# Cell 4 — Merge batch03 + model3 train/val into clean YOLO structure
import shutil
from pathlib import Path

ROOT = Path("/kaggle/working/plates_model4")
TI   = ROOT / "images/train"
TL   = ROOT / "labels/train"
VI   = ROOT / "images/validation"
VL   = ROOT / "labels/validation"
for d in [TI, TL, VI, VL]:
    d.mkdir(parents=True, exist_ok=True)

BASE = Path("/kaggle/input/datasets/abdullahaamir13/numberplatesmodel03")

# ── SOURCE 1: batch03 obj_train_data (flat jpg+txt pairs) ────────────────────
BATCH03 = BASE / "batch03-model3/obj_train_data"
bus_pos = bus_neg = 0
for img in BATCH03.glob("*.jpg"):
    lbl = img.with_suffix(".txt")
    if not lbl.exists():
        continue
    shutil.copy(img, TI / f"bus_{img.name}")
    shutil.copy(lbl, TL / f"bus_{lbl.name}")
    if lbl.stat().st_size > 0:
        bus_pos += 1
    else:
        bus_neg += 1
print(f"[1/2] batch03: {bus_pos} with plates  |  {bus_neg} negatives")

# ── SOURCE 2: model3/model3/train+val  (images/ + labels/ subfolders) ─────────
def copy_split(split_root, img_dst, lbl_dst, prefix):
    img_src = split_root / "images"
    lbl_src = split_root / "labels"
    added = 0
    for img in img_src.glob("*.jpg"):
        lbl = lbl_src / (img.stem + ".txt")
        if lbl.exists():
            shutil.copy(img, img_dst / f"{prefix}_{img.name}")
            shutil.copy(lbl, lbl_dst / f"{prefix}_{lbl.name}")
            added += 1
    return added

n_train = copy_split(BASE / "model3/model3/train", TI, TL, "m3")
n_val   = copy_split(BASE / "model3/model3/val",   VI, VL, "m3v")
print(f"[2/2] model3 splits: train={n_train}  |  val={n_val}")

print(f"\nTotal train : {len(list(TI.glob('*.jpg'))):,}")
print(f"Total val   : {len(list(VI.glob('*.jpg'))):,}")

[1/2] batch03: 993 with plates  |  126 negatives
[2/2] model3 splits: train=475  |  val=119

Total train : 1,594
Total val   : 119


In [2]:
# Cell 5 — Auto-split 10% val from train if model3 val was empty
import shutil, random
from pathlib import Path

TI = Path("/kaggle/working/plates_model4/images/train")
TL = Path("/kaggle/working/plates_model4/labels/train")
VI = Path("/kaggle/working/plates_model4/images/validation")
VL = Path("/kaggle/working/plates_model4/labels/validation")

val_count = len(list(VI.glob("*.jpg")))
if val_count == 0:
    all_imgs = list(TI.glob("*.jpg"))
    random.seed(42)
    random.shuffle(all_imgs)
    split_n = max(1, int(len(all_imgs) * 0.10))
    for img in all_imgs[:split_n]:
        lbl = TL / (img.stem + ".txt")
        shutil.move(str(img), VI / img.name)
        if lbl.exists():
            shutil.move(str(lbl), VL / lbl.name)
    print(f"✅ Auto-split: {split_n} → val  |  {len(all_imgs)-split_n} → train")
else:
    print(f"✅ Val already populated: {val_count} images — no split needed")

print(f"Final → train: {len(list(TI.glob('*.jpg'))):,}  |  val: {len(list(VI.glob('*.jpg'))):,}")

✅ Val already populated: 119 images — no split needed
Final → train: 1,594  |  val: 119


In [24]:
# Cell 6 — Validate all labels before training
from pathlib import Path

errors = []
for split, img_dir, lbl_dir in [
    ("train", Path("/kaggle/working/plates_model4/images/train"),
               Path("/kaggle/working/plates_model4/labels/train")),
    ("val",   Path("/kaggle/working/plates_model4/images/validation"),
               Path("/kaggle/working/plates_model4/labels/validation")),
]:
    for img in img_dir.glob("*.jpg"):
        lbl = lbl_dir / (img.stem + ".txt")
        if not lbl.exists():
            errors.append(f"MISSING_LABEL: {img.name}")
            continue
        for ln, line in enumerate(lbl.read_text().strip().splitlines(), 1):
            if not line.strip():
                continue
            parts = line.split()
            if len(parts) != 5:
                errors.append(f"BAD_COLS({len(parts)}): {lbl.name}:L{ln}")
                continue
            cls_id = int(parts[0])
            if cls_id != 0:
                errors.append(f"WRONG_CLASS={cls_id}: {lbl.name}:L{ln}")
            for v, name in zip(parts[1:], ["cx","cy","bw","bh"]):
                fv = float(v)
                if not (0.0 <= fv <= 1.0):
                    errors.append(f"OUT_OF_RANGE {name}={fv:.4f}: {lbl.name}:L{ln}")

if errors:
    print(f"{len(errors)} label errors found (first 30):")
    for e in errors[:30]:
        print(f"   {e}")
else:
    print("All labels valid clean dataset ready for training!")

All labels valid clean dataset ready for training!


## Step 4 Train Model04

In [25]:
# Cell 7 — Write dataset.yaml
import yaml

config = {
    "path" : "/kaggle/working/plates_model4",
    "train": "images/train",
    "val"  : "images/validation",
    "nc"   : 1,
    "names": {0: "number_plate"},
}

with open("/kaggle/working/plates_model4/dataset.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

!cat /kaggle/working/plates_model4/dataset.yaml

names:
  0: number_plate
nc: 1
path: /kaggle/working/plates_model4
train: images/train
val: images/validation


In [26]:
# Cell 8 — Fine-tune from Model03 weights
# Starting from Model03 (mAP50=0.991) means we only need 30 epochs
# to adapt to the corrected bus plate data

MODEL03_PT = "/kaggle/input/datasets/abdullahaamir13/plates-model03-results/weights/best.pt"

!yolo task=detect \
      mode=train \
      model={MODEL03_PT} \
      data=/kaggle/working/plates_model4/dataset.yaml \
      epochs=30 \
      imgsz=640 \
      batch=16 \
      patience=10 \
      name=plates_model4 \
      project=/kaggle/working/runs \
      device=0 \
      exist_ok=True \
      cos_lr=True \
      mosaic=1.0 \
      lr0=0.0005 \
      lrf=0.01 \
      warmup_epochs=2 \
      save_period=5

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/kaggle/working/plates_model4/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/input/datasets/abdullahaamir13/plates-model03-results/weights/best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plates_model4, nbs=64, nms

In [28]:
# Cell 9 — Compare Model03 vs Model04
from ultralytics import YOLO

DATA = "/kaggle/working/plates_model4/dataset.yaml"

print("=" * 55)
print("▶ MODEL03 (baseline)")
m3 = YOLO("/kaggle/input/datasets/abdullahaamir13/plates-model03-results/weights/best.pt")
r3 = m3.val(data=DATA, imgsz=640, device=0, verbose=False)
print(f"  Precision : {r3.box.mp:.4f}")
print(f"  Recall    : {r3.box.mr:.4f}")
print(f"  mAP@50    : {r3.box.map50:.4f}")
print(f"  mAP@50-95 : {r3.box.map:.4f}")

print("=" * 55)
print("▶ MODEL04 (fine-tuned with corrected bus data)")
m4 = YOLO("/kaggle/working/runs/plates_model4/weights/best.pt")
r4 = m4.val(data=DATA, imgsz=640, device=0, verbose=False)
print(f"  Precision : {r4.box.mp:.4f}")
print(f"  Recall    : {r4.box.mr:.4f}")
print(f"  mAP@50    : {r4.box.map50:.4f}")
print(f"  mAP@50-95 : {r4.box.map:.4f}")

print("=" * 55)
print(f"  Δ mAP@50    : {r4.box.map50 - r3.box.map50:+.4f}")
print(f"  Δ mAP@50-95 : {r4.box.map   - r3.box.map:+.4f}")

▶ MODEL03 (baseline)
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 920.2±381.9 MB/s, size: 24.5 KB)
val: Scanning /kaggle/working/plates_model4/labels/validation.cache... 2003 images, 106 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 2003/2003 336.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 75% ━━━━━━━━━─── 95/126 9.4it/s 10.4s<3.3s

Corrupt JPEG data: 2 extraneous bytes before marker 0xd9
Corrupt JPEG data: 2 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9
Corrupt JPEG data: 2 extraneous bytes before marker 0xd9
Corrupt JPEG data: 2 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9
Corrupt JPEG data: 2 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 77% ━━━━━━━━━─── 97/126 9.1it/s 10.6s<3.2s

Corrupt JPEG data: 2 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 84% ━━━━━━━━━━── 106/126 8.5it/s 11.7s<2.3s

...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 86% ━━━━━━━━━━── 108/126 8.3it/s 12.1s<2.2s

Corrupt JPEG data: 2 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 88% ━━━━━━━━━━╸─ 111/126 7.7it/s 12.5s<1.9s

Corrupt JPEG data: 1 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 126/126 8.9it/s 14.2s0.1s
                   all       2003       1976      0.966      0.969      0.985      0.722
Speed: 0.6ms preprocess, 3.6ms inference, 0.0ms loss, 0.7ms postprocess per image
Results saved to /kaggle/working/runs/detect/val2
  Precision : 0.9657
  Recall    : 0.9687
  mAP@50    : 0.9850
  mAP@50-95 : 0.7222
  Δ mAP@50    : +0.0076
  Δ mAP@50-95 : +0.0128


In [34]:
# Cell 10 — Save Model04 weights
import shutil
shutil.make_archive("/kaggle/working/plates_model4_weights", "zip",
                    "/kaggle/working/runs/plates_model4/weights")
print(" plates_model4_weights.zip ready in Output tab")
print("   best.pt  ← use this for inference")

 plates_model4_weights.zip ready in Output tab
   best.pt  ← use this for inference


## Step 4 OCR Pipeline

`Image → YOLOv8 detect plate → crop → preprocess → EasyOCR → text`

In [16]:
# Cell 11 — Load detection model + OCR engine (run once)
import easyocr
from ultralytics import YOLO

detector = YOLO("/kaggle/input/datasets/abdullahaamir13/platesfinalresult/best.pt")
reader   = easyocr.Reader(["en"], gpu=True, verbose=False)

print("YOLOv8 Model04 loaded")
print("EasyOCR loaded")

YOLOv8 Model04 loaded
EasyOCR loaded


In [17]:
# Cell 12 — Preprocessing + OCR helpers
import cv2, numpy as np, re

def preprocess_plate_crop(crop: np.ndarray) -> np.ndarray:
    """Upscale → grayscale → denoise → adaptive threshold → sharpen"""
    h, w = crop.shape[:2]
    if w < 300:
        crop = cv2.resize(crop, (300, int(h * 300/w)), interpolation=cv2.INTER_CUBIC)
    gray   = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
    gray   = cv2.fastNlMeansDenoising(gray, h=10)
    thresh = cv2.adaptiveThreshold(gray, 255,
                cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 31, 10)
    kernel = np.array([[0,-1,0],[-1,5,-1],[0,-1,0]])
    return cv2.filter2D(thresh, -1, kernel)

def clean_plate_text(raw: str) -> str:
    """Strip noise, uppercase — keeps alphanumeric and hyphen"""
    return re.sub(r"[^A-Z0-9\-]", "", raw.upper()).strip()

print("Helpers defined")

Helpers defined


In [18]:
# Cell 13 — End-to-end detect + OCR function
import cv2, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

def detect_and_read_plate(image_path: str,
                           conf: float = 0.35,
                           show: bool = True) -> list:
    """
    Full pipeline: image → detect plates → crop → OCR → text
    Returns: [{'bbox':[x1,y1,x2,y2], 'conf':float, 'text':str, 'raw':str}]
    """
    img_bgr = cv2.imread(image_path)
    if img_bgr is None:
        raise FileNotFoundError(image_path)
    H, W = img_bgr.shape[:2]

    results = []
    for r in detector.predict(source=image_path, conf=conf, iou=0.45,
                               imgsz=640, verbose=False, stream=True):
        for box in (r.boxes or []):
            x1,y1,x2,y2 = map(int, box.xyxy[0].tolist())
            det_conf     = float(box.conf[0])

            # Pad 5%
            px=int((x2-x1)*0.05); py=int((y2-y1)*0.05)
            crop = img_bgr[max(0,y1-py):min(H,y2+py),
                           max(0,x1-px):min(W,x2+px)]

            processed = preprocess_plate_crop(crop)
            ocr_out   = reader.readtext(processed, detail=1, paragraph=False)
            raw_text  = " ".join(seg[1] for seg in sorted(ocr_out, key=lambda s: s[0][0][0]))
            clean     = clean_plate_text(raw_text)

            results.append({"bbox":[x1,y1,x2,y2], "conf":det_conf,
                             "text":clean, "raw":raw_text})

    if show and results:
        vis = cv2.cvtColor(img_bgr.copy(), cv2.COLOR_BGR2RGB)
        for d in results:
            x1,y1,x2,y2 = d["bbox"]
            label = f"{d['text']}  {d['conf']:.0%}"
            cv2.rectangle(vis,(x1,y1),(x2,y2),(0,220,0),3)
            cv2.rectangle(vis,(x1,y1-34),(x1+len(label)*13,y1),(0,220,0),-1)
            cv2.putText(vis, label,(x1+4,y1-8),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9,(0,0,0),2)
        plt.figure(figsize=(14,7))
        plt.imshow(vis); plt.axis("off")
        plt.title(f"{len(results)} plate(s) detected  |  {Path(image_path).name}", fontsize=13)
        plt.tight_layout(); plt.show()

    return results

print("detect_and_read_plate() ready")

detect_and_read_plate() ready


In [20]:
# Cell 14 — Test on one image from batch03
from pathlib import Path

# Pick first image from batch03 (has ground truth so easy to verify)
test_imgs = sorted(
    Path("/kaggle/input/datasets/abdullahaamir13/numberplatesmodel03/batch03-model3/obj_train_data")
    .glob("*.jpg")
)

if test_imgs:
    img = str(test_imgs[0])
    print(f"Test image: {img}\n")
    results = detect_and_read_plate(img, conf=0.35, show=True)
    for i, r in enumerate(results, 1):
        print(f"Plate #{i}  conf={r['conf']:.1%}  text='{r['text']}'  raw='{r['raw']}'")
else:
    print("No images found — check BATCH03 path in Cell 2.")

Test image: /kaggle/input/datasets/abdullahaamir13/numberplatesmodel03/batch03-model3/obj_train_data/01000_bus_0.jpg



## Step 5 Batch Inference → CSV

In [21]:
# Cell 15 — Run OCR on all batch03 images, save to CSV
import csv
from pathlib import Path

INPUT_DIR  = Path("/kaggle/input/datasets/abdullahaamir13/numberplatesmodel03/batch03-model3/obj_train_data")
OUTPUT_CSV = "/kaggle/working/plate_ocr_results.csv"
CONF       = 0.35

image_paths = sorted(INPUT_DIR.glob("*.jpg"))
print(f"Processing {len(image_paths)} images...")

rows = []
for idx, img_path in enumerate(image_paths, 1):
    try:
        dets = detect_and_read_plate(str(img_path), conf=CONF, show=False)
        if dets:
            for d in dets:
                rows.append({"image":img_path.name, "plate":d["text"],
                             "raw_ocr":d["raw"], "conf":round(d["conf"],3),
                             "x1":d["bbox"][0],"y1":d["bbox"][1],
                             "x2":d["bbox"][2],"y2":d["bbox"][3]})
        else:
            rows.append({"image":img_path.name,"plate":"","raw_ocr":"",
                         "conf":0,"x1":0,"y1":0,"x2":0,"y2":0})
    except Exception as e:
        rows.append({"image":img_path.name,"plate":f"ERROR:{e}",
                     "raw_ocr":"","conf":0,"x1":0,"y1":0,"x2":0,"y2":0})
    if idx % 100 == 0:
        print(f"  [{idx}/{len(image_paths)}]...")

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["image","plate","raw_ocr","conf","x1","y1","x2","y2"])
    writer.writeheader()
    writer.writerows(rows)

detected = sum(1 for r in rows if r["plate"] and not r["plate"].startswith("ERROR"))
print(f"\n Total: {len(image_paths)}  |  Plates read: {detected}  |  Rate: {detected/len(image_paths)*100:.1f}%")
print(f"    CSV → {OUTPUT_CSV}")

Processing 1119 images...
  [100/1119]...
  [200/1119]...
  [300/1119]...
  [400/1119]...
  [500/1119]...


Corrupt JPEG data: 1 extraneous bytes before marker 0xd9
Corrupt JPEG data: 1 extraneous bytes before marker 0xd9


  [600/1119]...
  [700/1119]...
  [800/1119]...
  [900/1119]...
  [1000/1119]...
  [1100/1119]...

 Total: 1119  |  Plates read: 755  |  Rate: 67.5%
    CSV → /kaggle/working/plate_ocr_results.csv


In [23]:
# Cell 16 — Preview results
import pandas as pd
df = pd.read_csv("/kaggle/working/plate_ocr_results.csv")
print(f"With plate text : {(df['plate']!='').sum()}")
print(f"Empty / no plate: {(df['plate']=='').sum()}")
display(df[df["plate"]!=""].head(20))

With plate text : 1218
Empty / no plate: 0


,image,plate,raw_ocr,conf,x1,y1,x2,y2
0,01000_bus_0.jpg,NaN,NaN,0.000,0,0,0,0
1,01001_bus_0.jpg,C4177,C4?177,0.788,444,740,515,781
2,01002_bus_0.jpg,NaN,NaN,0.000,0,0,0,0
3,01003_bus_0.jpg,NaN,NaN,0.768,296,236,315,249
4,01004_bus_0.jpg,CAJ-7795BE,CAJ-7795 ~be,0.822,668,1211,831,1263
5,01005_bus_0.jpg,NaN,NaN,0.847,208,385,233,397
6,01005_bus_1.jpg,NaN,NaN,0.000,0,0,0,0
7,01006_bus_0.jpg,ENELI,ENELI,0.771,958,947,1033,979
8,01007_bus_0.jpg,ISCP,ISC; (}P,0.819,320,563,357,580
9,01007_bus_1.jpg,NaN,NaN,0.000,0,0,0,0


## Step 6 Export

In [26]:
import shutil, os
from ultralytics import YOLO
from IPython.display import FileLink

# Step 1: Copy pt to working dir first
shutil.copy(
    "/kaggle/input/datasets/abdullahaamir13/platesfinalresult/best.pt",
    "/kaggle/working/best_model4.pt"
)
print("Copied to working dir")

# Step 2: Load FROM working dir — export will save next to it
m4 = YOLO("/kaggle/working/best_model4.pt")
m4.export(format="onnx", imgsz=640, dynamic=False, simplify=True, opset=12)
print("ONNX exported → /kaggle/working/best_model4.onnx")

# Step 3: Zip both
shutil.make_archive("/kaggle/working/plates_model4_weights", "zip",
                    "/kaggle/working", "best_model4.pt")

print("\n Files:")
for f in ["best_model4.pt", "best_model4.onnx", "plates_model4_weights.zip"]:
    p = f"/kaggle/working/{f}"
    if os.path.exists(p):
        print(f"  {f}  —  {os.path.getsize(p)/1e6:.1f} MB")

display(FileLink("/kaggle/working/plates_model4_weights.zip"))
display(FileLink("/kaggle/working/best_model4.onnx"))

Copied to working dir
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.9.0+cu126 CPU (Intel Xeon CPU @ 2.00GHz)
Model summary (fused): 73 layers, 11,125,971 parameters, 0 gradients, 28.4 GFLOPs

PyTorch: starting from '/kaggle/working/best_model4.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 5, 8400) (21.5 MB)

ONNX: starting export with onnx 1.20.1 opset 12...
ONNX: slimming with onnxslim 0.1.86...
ONNX: export success ✅ 1.7s, saved as '/kaggle/working/best_model4.onnx' (42.7 MB)

Export complete (2.3s)
Results saved to /kaggle/working
Predict:         yolo predict task=detect model=/kaggle/working/best_model4.onnx imgsz=640 
Validate:        yolo val task=detect model=/kaggle/working/best_model4.onnx imgsz=640 data=/kaggle/working/plates_model4/dataset.yaml  
Visualize:       https://netron.app
ONNX exported → /kaggle/working/best_model4.onnx

 Files:
  best_model4.pt  —  22.5 MB
  best_model4.onnx  —  44.7 MB
  plates_model4_weights.zip  —  20.7 MB


/kaggle/working/plates_model4_weights.zip

/kaggle/working/best_model4.onnx

## Quick-Use Inference (copy anywhere)

```python
from ultralytics import YOLO
import easyocr, cv2, re

detector = YOLO("best.pt")
reader   = easyocr.Reader(["en"], gpu=False)

def plate_ocr(image_path, conf=0.35):
    preds = detector.predict(image_path, conf=conf, verbose=False)
    img   = cv2.imread(image_path)
    out   = []
    for box in (preds[0].boxes or []):
        x1,y1,x2,y2 = map(int, box.xyxy[0])
        gray = cv2.cvtColor(img[y1:y2, x1:x2], cv2.COLOR_BGR2GRAY)
        text = re.sub(r"[^A-Z0-9-]", "", " ".join(
            r[1] for r in easyocr.Reader(["en"]).readtext(gray, detail=0)
        ).upper())
        out.append({"plate": text, "conf": float(box.conf[0]), "bbox": [x1,y1,x2,y2]})
    return out

for r in plate_ocr("image.jpg"):
    print(r["plate"], f"({r['conf']:.0%})")
```
